In [1]:
import os

from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from array import array
import os
from PIL import Image
import sys
import time
import matplotlib.pyplot as plt
import numpy as np

'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

# 1. Sa se foloseasca un algoritm de clasificare a imaginilor (etapa de inferenta/testare) si sa se stabileasca performanta acestui algoritm de clasificare binara (imagini cu biciclete vs. imagini fara biciclete).

In [2]:
def detect_bicycle(img_path):
    with open(img_path, "rb") as img:
        result = computervision_client.analyze_image_in_stream(img, visual_features=[VisualFeatureTypes.tags, VisualFeatureTypes.objects])

        for tag in result.tags:
            if tag.name == "bicycle":
                return True

        return False

In [3]:
def classify_image(img_path):
    return detect_bicycle(img_path) # Returns True if a bicycle is detected, otherwise returns False

In [4]:
def process_images(folder_path):
    results = {}
    for img in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img)
        result = classify_image(img_path)
        results[img] = result
    return results

In [12]:
def compute_metrics(predictions, ground_truth):
    TP = FP = TN = FN = 0
    for img in ground_truth:
        pred = predictions.get(img, False)
        actual = ground_truth[img]
        if actual == True and pred == True:
            TP += 1
        elif actual == False and pred == True:
            FP += 1
        elif actual == False and pred == False:
            TN += 1
        elif actual == True and pred == False:
            FN += 1

    print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")

    accuracy = (TP + TN) / (TP + FP + TN + FN) if (TP + FP + TN + FN) > 0 else 0 # cat de des are dreptate modelul in general, adica din toate predictiile, cate sunt corecte
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0 # cat de des are dreptate modelul atunci cand prezice ca e bicicleta, adica din toate predictiile pozitive, cate sunt corecte
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0 # cate biciclete reale a detectat modelul, adica din toate bicicletele reale, cate a detectat corect
    return accuracy, precision, recall

In [9]:
ground_truth = {
    "bike1.jpg": True,
    "bike02.jpg": True,
    "bike03.jpg": True,
    "bike04.jpg": True,
    "bike05.jpg": True,
    "bike06.jpg": True,
    "bike07.jpg": True,
    "bike08.jpg": True,
    "bike09.jpg": True,
    "bike10.jpg": True,
    "traffic01.jpg": False,
    "traffic02.jpg": False,
    "traffic03.jpg": False,
    "traffic04.jpg": False,
    "traffic05.jpg": False,
    "traffic06.jpg": False,
    "traffic07.jpg": False,
    "traffic08.jpg": False,
    "traffic09.jpg": False,
    "traffic10.jpg": False,
}

In [13]:
predictions = process_images("images/bikes")
accuracy, precision, recall = compute_metrics(predictions, ground_truth)
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

TP: 10, FP: 0, TN: 10, FN: 0
Accuracy: 1.00
Precision: 1.00
Recall: 1.00
